In [62]:
import os
import sys
import numpy as np
import re
import toml

In [108]:
config_file = "config.toml"
config = toml.load(config_file)

npz_dir = config["coverage_splitting"]["npz_dir"]
export_dir = config["coverage_splitting"]["export_dir"]
threshold = config["coverage_splitting"]["threshold"]
rounding_decimals = config["coverage_splitting"]["rounding_decimals"]

In [88]:
for filename in os.listdir(npz_dir):
    if filename.endswith(".npz"):
        break

filename = os.path.join(npz_dir, filename)
file = np.load(filename, allow_pickle=True)
print(file.keys())
print(len(np.where(np.diff(file["timestamps"]) > threshold)[0]))

patterns = []
for pattern in file["patterns"]:
    patterns.append(np.unpackbits(np.array(pattern, dtype=np.uint8), bitorder="big"))
patterns = np.array(patterns)
print(patterns.shape)

KeysView(NpzFile 'exp_data/led_frozen_noise\\2025-08-07T09-14-29_RecID-1016_45581_Typ4-18h-1N-14p4I-2N_pois008-025_5p2V-5p5V-6V_DIV21_HS277_only-de_corrected.npz' with keys: timestamps, patterns)
95
(415615, 256)


In [ ]:
# For each block, get average coverage and save identical coverages to new .npz file
def split_file(filepath):
    file = np.load(filepath, allow_pickle=True)
    # Get all blocks
    blocks = np.where(np.diff(file["timestamps"]) > threshold)[0]
    print(f"Found {len(blocks)+1} blocks in {filepath}.")  # +1 because 0 is also a block start
    
    grouped_patterns = {}
    grouped_timestamps = {}
    
    # Get average coverage in each block
    for i in range(len(blocks) + 1):
        # Get block patterns
        if i == 0:
            start = 0
            end = blocks[i]
        elif i == len(blocks):
            start = blocks[i - 1]
            end = len(file["patterns"])
        else:
            start = blocks[i - 1]
            end = blocks[i]

        block_patterns = file["patterns"][start:end]
        block_timestamps = file["timestamps"][start:end]

        # Unpack all patterns
        unpacked = np.array([np.unpackbits(np.array(pattern, dtype=np.uint8), bitorder="big") for pattern in block_patterns])
        # and average coverage, rounding to 2 decimal places
        avg_coverage = np.round(np.mean(unpacked), decimals = rounding_decimals)
        coverage_str = f"{int(avg_coverage * 100):03d}"
        print(f"Block {i}: {avg_coverage} -> {coverage_str}")
        
        if coverage_str not in grouped_patterns:
            grouped_patterns[coverage_str] = []
            grouped_timestamps[coverage_str] = []

        grouped_patterns[coverage_str].append(block_patterns)
        grouped_timestamps[coverage_str].append(block_timestamps)

    base_filename = os.path.basename(filepath)

    for coverage_str in grouped_patterns:
        patterns = np.concatenate(grouped_patterns[coverage_str])
        timestamps = np.concatenate(grouped_timestamps[coverage_str])

        # Replace poisXXX or poisXXX-YYY with split_poisXXX
        new_filename = re.sub(r"pois\d{3}(?:-\d{3})?", f"split_pois{coverage_str}", base_filename)
        output_path = os.path.join(export_dir, new_filename)
        np.savez(output_path, patterns=patterns, timestamps=timestamps)

    print(f"Saved: {output_path}")

In [112]:
for filename in os.listdir(npz_dir):
    if filename.endswith(".npz"):
        split_file(os.path.join(npz_dir, filename))


Found 96 blocks in exp_data/led_frozen_noise\2025-08-07T09-14-29_RecID-1016_45581_Typ4-18h-1N-14p4I-2N_pois008-025_5p2V-5p5V-6V_DIV21_HS277_only-de_corrected.npz.
Block 0: 0.08 -> 008
Block 1: 0.24 -> 024
Block 2: 0.08 -> 008
Block 3: 0.24 -> 024
Block 4: 0.08 -> 008
Block 5: 0.24 -> 024
Block 6: 0.08 -> 008
Block 7: 0.24 -> 024
Block 8: 0.08 -> 008
Block 9: 0.24 -> 024
Block 10: 0.08 -> 008
Block 11: 0.24 -> 024
Block 12: 0.08 -> 008
Block 13: 0.24 -> 024


KeyboardInterrupt: 

In [ ]:
import numpy as np

path = "E:\Git\RFArduinoData\example_scripts\exp_data\led_halffield\detailed_sync_2025-08-13T16-42-35_RecID-1034_45581_Typ2-Blocked-HalfField-0p25I-4x0p12N-0p12I_pois025_6V_DIV27_HS257_only-de_trimmed_2025-10-10_01-22-12.npz"
data = np.load(path, allow_pickle=True)

<>:3: SyntaxWarning: invalid escape sequence '\G'
<>:3: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Laslo\AppData\Local\Temp\ipykernel_31488\890237175.py:3: SyntaxWarning: invalid escape sequence '\G'
  path = "E:\Git\RFArduinoData\example_scripts\exp_data\led_halffield\detailed_sync_2025-08-13T16-42-35_RecID-1034_45581_Typ2-Blocked-HalfField-0p25I-4x0p12N-0p12I_pois025_6V_DIV27_HS257_only-de_trimmed_2025-10-10_01-22-12.npz"


In [16]:
(data["timestamps"][-1] - data["timestamps"][0]) / (60*1e6)

15.0